In [0]:
# Read the table directly from your Databricks catalog workspace
raw_df = spark.read.table("workspace.default.online_retail")

# View the structure and the first few records
raw_df.printSchema()
display(raw_df.limit(5))

root
 |-- InvoiceNo: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: long (nullable = true)
 |-- InvoiceDate: timestamp (nullable = true)
 |-- UnitPrice: double (nullable = true)
 |-- CustomerID: double (nullable = true)
 |-- Country: string (nullable = true)



InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01T08:26:00.000Z,2.55,17850.0,United Kingdom
536365,71053,WHITE METAL LANTERN,6,2010-12-01T08:26:00.000Z,3.39,17850.0,United Kingdom
536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01T08:26:00.000Z,2.75,17850.0,United Kingdom
536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01T08:26:00.000Z,3.39,17850.0,United Kingdom
536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01T08:26:00.000Z,3.39,17850.0,United Kingdom


In [0]:
from pyspark.sql.functions import col, to_date

# 1. Strip missing values from target columns
cleaned_df = raw_df.dropna(subset=["CustomerID", "Description"])

# 2. Filter out negative inventory metrics and transaction noise
cleaned_df = cleaned_df.filter((col("Quantity") > 0) & (col("UnitPrice") > 0))

# 3. Create a financial transaction spending matrix
cleaned_df = cleaned_df.withColumn("TotalSpending", col("Quantity") * col("UnitPrice"))

display(cleaned_df.limit(5))

InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TotalSpending
536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01T08:26:00.000Z,2.55,17850.0,United Kingdom,15.299999999999999
536365,71053,WHITE METAL LANTERN,6,2010-12-01T08:26:00.000Z,3.39,17850.0,United Kingdom,20.34
536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01T08:26:00.000Z,2.75,17850.0,United Kingdom,22.0
536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01T08:26:00.000Z,3.39,17850.0,United Kingdom,20.34
536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01T08:26:00.000Z,3.39,17850.0,United Kingdom,20.34


In [0]:
# Run aggregation logic to identify high-value consumer groups
customer_metrics_df = cleaned_df.groupBy("CustomerID") \
    .agg(
        {"TotalSpending": "sum", "Quantity": "sum"} \
    ).withColumnRenamed("sum(TotalSpending)", "TotalAmountSpent") \
     .withColumnRenamed("sum(Quantity)", "TotalItemsBought") \
     .orderBy(col("TotalAmountSpent").desc())

# Save your final data asset straight back to Delta Lake format
customer_metrics_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.gold_customer_metrics")

print("Pipeline completed successfully! Portfolio data asset created.")

Pipeline completed successfully! Portfolio data asset created.
